In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path("../data/raw")

collisions = pd.read_csv(DATA_PATH / "collisions.csv")
vehicles = pd.read_csv(DATA_PATH / "vehicles.csv")
casualties = pd.read_csv(DATA_PATH / "casualties.csv")

print("Raw datasets loaded successfully!")
print("Collisions:", collisions.shape)
print("Vehicles:", vehicles.shape)
print("Casualties:", casualties.shape)

C:\Users\shiva\AppData\Local\Temp\ipykernel_16136\1550745128.py:8: DtypeWarning: Columns (27: generic_make_model) have mixed types. Specify dtype option on import or set low_memory=False.
  vehicles = pd.read_csv(DATA_PATH / "vehicles.csv")


Raw datasets loaded successfully!
Collisions: (101525, 44)
Vehicles: (183948, 32)
Casualties: (127883, 24)


In [2]:
collisions_clean = collisions.copy()
vehicles_clean = vehicles.copy()
casualties_clean = casualties.copy()

print("Working copies created successfully!")

print("Collisions:", collisions_clean.shape)
print("Vehicles:", vehicles_clean.shape)
print("Casualties:", casualties_clean.shape)

Working copies created successfully!
Collisions: (101525, 44)
Vehicles: (183948, 32)
Casualties: (127883, 24)


In [3]:
collisions_clean["date"] = pd.to_datetime(
    collisions_clean["date"],
    format="%d/%m/%Y",
    errors="coerce"
)

print("Date conversion completed!")
print("New data type:", collisions_clean["date"].dtype)
print("Missing dates after conversion:", collisions_clean["date"].isna().sum())

print("\nFirst 5 converted dates:")
print(collisions_clean["date"].head())

Date conversion completed!
New data type: datetime64[us]
Missing dates after conversion: 0

First 5 converted dates:
0   2025-02-04
1   2025-05-07
2   2025-11-27
3   2025-05-15
4   2025-12-08
Name: date, dtype: datetime64[us]


In [4]:
collisions_clean["year"] = collisions_clean["date"].dt.year
collisions_clean["month"] = collisions_clean["date"].dt.month
collisions_clean["month_name"] = collisions_clean["date"].dt.month_name()
collisions_clean["day_name"] = collisions_clean["date"].dt.day_name()
collisions_clean["quarter"] = collisions_clean["date"].dt.quarter

print("Date features created successfully!")

print(
    collisions_clean[
        ["date", "year", "month", "month_name", "day_name", "quarter"]
    ].head()
)

Date features created successfully!
        date  year  month month_name   day_name  quarter
0 2025-02-04  2025      2   February    Tuesday        1
1 2025-05-07  2025      5        May  Wednesday        2
2 2025-11-27  2025     11   November   Thursday        4
3 2025-05-15  2025      5        May   Thursday        2
4 2025-12-08  2025     12   December     Monday        4


In [5]:
collisions_clean["time_clean"] = pd.to_datetime(
    collisions_clean["time"],
    format="%H:%M",
    errors="coerce"
)

print("Time conversion completed!")
print("Missing times before conversion:", collisions_clean["time"].isna().sum())
print("Invalid/missing times after conversion:", collisions_clean["time_clean"].isna().sum())

print("\nFirst 5:")
print(collisions_clean[["time", "time_clean"]].head())

Time conversion completed!
Missing times before conversion: 0
Invalid/missing times after conversion: 0

First 5:
    time          time_clean
0  15:00 1900-01-01 15:00:00
1  09:35 1900-01-01 09:35:00
2  18:18 1900-01-01 18:18:00
3  16:30 1900-01-01 16:30:00
4  07:45 1900-01-01 07:45:00


In [6]:
# Extract hour from the converted time
collisions_clean["hour"] = collisions_clean["time_clean"].dt.hour

# Create time-of-day categories
def get_time_of_day(hour):
    if 5 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 17:
        return "Afternoon"
    elif 17 <= hour < 21:
        return "Evening"
    else:
        return "Night"

collisions_clean["time_of_day"] = collisions_clean["hour"].apply(get_time_of_day)

print("Time features created successfully!")

print(
    collisions_clean[
        ["time", "hour", "time_of_day"]
    ].head(10)
)

Time features created successfully!
    time  hour time_of_day
0  15:00    15   Afternoon
1  09:35     9     Morning
2  18:18    18     Evening
3  16:30    16   Afternoon
4  07:45     7     Morning
5  12:11    12   Afternoon
6  11:45    11     Morning
7  17:31    17     Evening
8  16:00    16   Afternoon
9  16:40    16   Afternoon


In [7]:
# STATS19 day_of_week:
# 1 = Sunday, 2 = Monday, ..., 7 = Saturday

day_map = {
    1: "Sunday",
    2: "Monday",
    3: "Tuesday",
    4: "Wednesday",
    5: "Thursday",
    6: "Friday",
    7: "Saturday"
}

collisions_clean["reported_day_name"] = (
    collisions_clean["day_of_week"].map(day_map)
)

day_mismatches = (
    collisions_clean["reported_day_name"]
    != collisions_clean["day_name"]
)

print("Total collisions:", len(collisions_clean))
print("Day-of-week mismatches:", day_mismatches.sum())

print("\nSample:")
print(
    collisions_clean[
        ["date", "day_of_week", "reported_day_name", "day_name"]
    ].head(10)
)

Total collisions: 101525
Day-of-week mismatches: 0

Sample:
        date  day_of_week reported_day_name   day_name
0 2025-02-04            3           Tuesday    Tuesday
1 2025-05-07            4         Wednesday  Wednesday
2 2025-11-27            5          Thursday   Thursday
3 2025-05-15            5          Thursday   Thursday
4 2025-12-08            2            Monday     Monday
5 2025-10-03            6            Friday     Friday
6 2025-07-25            6            Friday     Friday
7 2025-11-05            4         Wednesday  Wednesday
8 2025-11-19            4         Wednesday  Wednesday
9 2025-08-20            4         Wednesday  Wednesday


In [8]:
severity_map = {
    1: "Fatal",
    2: "Serious",
    3: "Slight"
}

collisions_clean["severity_label"] = (
    collisions_clean["collision_severity"].map(severity_map)
)

print("Severity labels created!")

print(
    collisions_clean[
        ["collision_severity", "severity_label"]
    ].value_counts().sort_index()
)

print(
    "\nUnmapped severity values:",
    collisions_clean["severity_label"].isna().sum()
)

Severity labels created!
collision_severity  severity_label
1                   Fatal              1453
2                   Serious           25191
3                   Slight            74881
Name: count, dtype: int64

Unmapped severity values: 0


In [9]:
vehicles_clean["age_of_driver_clean"] = (
    vehicles_clean["age_of_driver"].replace(-1, np.nan)
)

casualties_clean["age_of_casualty_clean"] = (
    casualties_clean["age_of_casualty"].replace(-1, np.nan)
)

print("Age cleaning completed!")

print("\nDRIVER AGE")
print("Original -1 values:",
      (vehicles_clean["age_of_driver"] == -1).sum())
print("Missing cleaned ages:",
      vehicles_clean["age_of_driver_clean"].isna().sum())

print("\nCASUALTY AGE")
print("Original -1 values:",
      (casualties_clean["age_of_casualty"] == -1).sum())
print("Missing cleaned ages:",
      casualties_clean["age_of_casualty_clean"].isna().sum())

Age cleaning completed!

DRIVER AGE
Original -1 values: 28262
Missing cleaned ages: 28262

CASUALTY AGE
Original -1 values: 2741
Missing cleaned ages: 2741


In [10]:
age_bins = [0, 16, 20, 25, 35, 45, 55, 65, 75, np.inf]

age_labels = [
    "0-15",
    "16-19",
    "20-24",
    "25-34",
    "35-44",
    "45-54",
    "55-64",
    "65-74",
    "75+"
]

vehicles_clean["driver_age_group"] = pd.cut(
    vehicles_clean["age_of_driver_clean"],
    bins=age_bins,
    labels=age_labels,
    right=False
)

print("Driver age groups created!")

print(
    vehicles_clean["driver_age_group"]
    .value_counts(dropna=False)
    .sort_index()
)

Driver age groups created!
driver_age_group
0-15      2348
16-19     9475
20-24    16109
25-34    34399
35-44    31121
45-54    24351
55-64    20518
65-74     9792
75+       7573
NaN      28262
Name: count, dtype: int64


In [11]:
casualties_clean["casualty_age_group"] = pd.cut(
    casualties_clean["age_of_casualty_clean"],
    bins=age_bins,
    labels=age_labels,
    right=False
)

print("Casualty age groups created!")

print(
    casualties_clean["casualty_age_group"]
    .value_counts(dropna=False)
    .sort_index()
)

Casualty age groups created!
casualty_age_group
0-15     11579
16-19    10820
20-24    13267
25-34    24761
35-44    20402
45-54    15669
55-64    13893
65-74     7658
75+       7093
NaN       2741
Name: count, dtype: int64


In [12]:
condition_columns = [
    "road_type",
    "speed_limit",
    "junction_detail",
    "light_conditions",
    "weather_conditions",
    "road_surface_conditions",
    "urban_or_rural_area"
]

for column in condition_columns:
    if column in collisions_clean.columns:
        values = sorted(
            collisions_clean[column]
            .dropna()
            .unique()
            .tolist()
        )

        print(f"{column}: {values}")
    else:
        print(f"{column}: COLUMN NOT FOUND")

road_type: [1, 2, 3, 6, 7, 9]
speed_limit: [20, 30, 40, 50, 60, 70]
junction_detail: [-1, 0, 13, 16, 17, 18, 19, 99]
light_conditions: [-1, 1, 4, 5, 6, 7]
weather_conditions: [1, 2, 3, 4, 5, 6, 7, 8, 9]
road_surface_conditions: [-1, 1, 2, 3, 4, 5, 9]
urban_or_rural_area: [1, 2, 3]


In [13]:
area_map = {
    1: "Urban",
    2: "Rural",
    3: "Unallocated"
}

collisions_clean["area_type"] = (
    collisions_clean["urban_or_rural_area"].map(area_map)
)

print("Area type labels created!")

print(
    collisions_clean[
        ["urban_or_rural_area", "area_type"]
    ].value_counts().sort_index()
)

print(
    "\nUnmapped area values:",
    collisions_clean["area_type"].isna().sum()
)

Area type labels created!
urban_or_rural_area  area_type  
1                    Urban          67148
2                    Rural          34373
3                    Unallocated        4
Name: count, dtype: int64

Unmapped area values: 0


In [14]:
road_type_map = {
    1: "Roundabout",
    2: "One way street",
    3: "Dual carriageway",
    6: "Single carriageway",
    7: "Slip road",
    9: "Unknown"
}

collisions_clean["road_type_label"] = (
    collisions_clean["road_type"].map(road_type_map)
)

print("Road type labels created!")

print(
    collisions_clean[
        ["road_type", "road_type_label"]
    ]
    .value_counts()
    .sort_index()
)

print(
    "\nUnmapped road type values:",
    collisions_clean["road_type_label"].isna().sum()
)

Road type labels created!
road_type  road_type_label   
1          Roundabout             6795
2          One way street         2398
3          Dual carriageway      14610
6          Single carriageway    74150
7          Slip road              1619
9          Unknown                1953
Name: count, dtype: int64

Unmapped road type values: 0


In [15]:
weather_map = {
    1: "Fine no high winds",
    2: "Raining no high winds",
    3: "Snowing no high winds",
    4: "Fine + high winds",
    5: "Raining + high winds",
    6: "Snowing + high winds",
    7: "Fog or mist",
    8: "Other",
    9: "Unknown"
}

collisions_clean["weather_label"] = (
    collisions_clean["weather_conditions"].map(weather_map)
)

print("Weather condition labels created!")

print(
    collisions_clean[
        ["weather_conditions", "weather_label"]
    ]
    .value_counts()
    .sort_index()
)

print(
    "\nUnmapped weather values:",
    collisions_clean["weather_label"].isna().sum()
)

Weather condition labels created!
weather_conditions  weather_label        
1                   Fine no high winds       83405
2                   Raining no high winds    10230
3                   Snowing no high winds      235
4                   Fine + high winds          715
5                   Raining + high winds       880
6                   Snowing + high winds        19
7                   Fog or mist                342
8                   Other                     3314
9                   Unknown                   2385
Name: count, dtype: int64

Unmapped weather values: 0


In [16]:
surface_map = {
    -1: "Data missing or out of range",
    1: "Dry",
    2: "Wet or damp",
    3: "Snow",
    4: "Frost or ice",
    5: "Flood over 3cm deep",
    9: "Unknown"
}

collisions_clean["road_surface_label"] = (
    collisions_clean["road_surface_conditions"].map(surface_map)
)

print("Road surface labels created!")

print(
    collisions_clean[
        ["road_surface_conditions", "road_surface_label"]
    ]
    .value_counts()
    .sort_index()
)

print(
    "\nUnmapped road surface values:",
    collisions_clean["road_surface_label"].isna().sum()
)

Road surface labels created!
road_surface_conditions  road_surface_label          
-1                       Data missing or out of range      634
 1                       Dry                             75679
 2                       Wet or damp                     22312
 3                       Snow                              146
 4                       Frost or ice                     1261
 5                       Flood over 3cm deep               151
 9                       Unknown                          1342
Name: count, dtype: int64

Unmapped road surface values: 0


In [17]:
light_map = {
    -1: "Data missing or out of range",
    1: "Daylight",
    4: "Darkness - lights lit",
    5: "Darkness - lights unlit",
    6: "Darkness - no lighting",
    7: "Darkness - lighting unknown"
}

collisions_clean["light_label"] = (
    collisions_clean["light_conditions"].map(light_map)
)

print("Light condition labels created!")

print(
    collisions_clean[
        ["light_conditions", "light_label"]
    ]
    .value_counts()
    .sort_index()
)

print(
    "\nUnmapped light values:",
    collisions_clean["light_label"].isna().sum()
)

Light condition labels created!
light_conditions  light_label                 
-1                Data missing or out of range       16
 1                Daylight                        72814
 4                Darkness - lights lit           20705
 5                Darkness - lights unlit           720
 6                Darkness - no lighting           5682
 7                Darkness - lighting unknown      1588
Name: count, dtype: int64

Unmapped light values: 0


In [18]:
def get_light_group(code):
    if code == 1:
        return "Daylight"
    elif code in [4, 5, 6, 7]:
        return "Darkness"
    else:
        return "Unknown / Missing"


collisions_clean["light_group"] = (
    collisions_clean["light_conditions"].apply(get_light_group)
)

print("Light groups created!")

print(
    collisions_clean["light_group"]
    .value_counts()
)

print(
    "\nTotal records:",
    collisions_clean["light_group"].value_counts().sum()
)

Light groups created!
light_group
Daylight             72814
Darkness             28695
Unknown / Missing       16
Name: count, dtype: int64

Total records: 101525


In [19]:
def get_peak_period(hour):
    if 7 <= hour < 10:
        return "Morning Peak"
    elif 16 <= hour < 19:
        return "Evening Peak"
    else:
        return "Off Peak"


collisions_clean["peak_period"] = (
    collisions_clean["hour"].apply(get_peak_period)
)

print("Peak period feature created!")

print(
    collisions_clean["peak_period"]
    .value_counts()
)

print(
    "\nTotal records:",
    collisions_clean["peak_period"].value_counts().sum()
)

Peak period feature created!
peak_period
Off Peak        61522
Evening Peak    24546
Morning Peak    15457
Name: count, dtype: int64

Total records: 101525


In [20]:
collisions_clean["day_type"] = np.where(
    collisions_clean["day_name"].isin(["Saturday", "Sunday"]),
    "Weekend",
    "Weekday"
)

print("Day type feature created!")

print(
    collisions_clean["day_type"]
    .value_counts()
)

print(
    "\nTotal records:",
    collisions_clean["day_type"].value_counts().sum()
)

Day type feature created!
day_type
Weekday    76673
Weekend    24852
Name: count, dtype: int64

Total records: 101525


In [21]:
print("COLLISION CLEANING VALIDATION")
print("-" * 40)

print("Rows:", len(collisions_clean))
print("Columns:", collisions_clean.shape[1])

print("\nDuplicate collision IDs:",
      collisions_clean["collision_index"].duplicated().sum())

print("Missing collision IDs:",
      collisions_clean["collision_index"].isna().sum())

print("Missing dates:",
      collisions_clean["date"].isna().sum())

print("Missing hours:",
      collisions_clean["hour"].isna().sum())

print("\nUnmapped severity:",
      collisions_clean["severity_label"].isna().sum())

print("Unmapped area type:",
      collisions_clean["area_type"].isna().sum())

print("Unmapped road type:",
      collisions_clean["road_type_label"].isna().sum())

print("Unmapped weather:",
      collisions_clean["weather_label"].isna().sum())

print("Unmapped road surface:",
      collisions_clean["road_surface_label"].isna().sum())

print("Unmapped light conditions:",
      collisions_clean["light_label"].isna().sum())

COLLISION CLEANING VALIDATION
----------------------------------------
Rows: 101525
Columns: 62

Duplicate collision IDs: 0
Missing collision IDs: 0
Missing dates: 0
Missing hours: 0

Unmapped severity: 0
Unmapped area type: 0
Unmapped road type: 0
Unmapped weather: 0
Unmapped road surface: 0
Unmapped light conditions: 0


In [22]:
vehicle_columns = [
    "vehicle_type",
    "sex_of_driver",
    "age_band_of_driver",
    "engine_capacity_cc",
    "age_of_vehicle",
    "journey_purpose_of_driver"
]

for column in vehicle_columns:
    if column in vehicles_clean.columns:
        values = sorted(
            vehicles_clean[column]
            .dropna()
            .unique()
            .tolist()
        )

        print(f"{column}: {values}")

    else:
        print(f"{column}: COLUMN NOT FOUND")

vehicle_type: [-1, 1, 2, 3, 4, 5, 8, 9, 10, 11, 16, 17, 18, 19, 20, 21, 22, 23, 33, 90, 97, 98, 99]
sex_of_driver: [-1, 1, 2, 3]
age_band_of_driver: [-1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
engine_capacity_cc: [-1, 1, 8, 12, 48, 49, 50, 75, 79, 85, 89, 96, 97, 98, 99, 100, 108, 109, 110, 113, 114, 115, 121, 123, 124, 125, 150, 153, 169, 171, 175, 179, 180, 185, 190, 195, 197, 198, 199, 200, 223, 225, 233, 234, 239, 244, 247, 248, 249, 250, 251, 263, 276, 278, 279, 286, 290, 292, 293, 295, 296, 299, 300, 309, 310, 312, 313, 321, 322, 330, 334, 346, 347, 348, 349, 350, 371, 373, 374, 378, 380, 385, 395, 398, 399, 400, 410, 411, 420, 421, 440, 448, 449, 450, 451, 452, 459, 470, 471, 482, 485, 487, 490, 492, 493, 494, 495, 496, 497, 498, 499, 500, 505, 510, 518, 519, 520, 530, 535, 550, 553, 562, 565, 567, 577, 582, 583, 588, 591, 592, 598, 599, 600, 601, 602, 608, 609, 613, 625, 636, 638, 644, 645, 646, 647, 648, 649, 650, 651, 652, 653, 654, 656, 659, 660, 670, 675, 676, 680, 686, 689, 69

In [23]:
sex_map = {
    -1: "Data missing or out of range",
    1: "Male",
    2: "Female",
    3: "Not known"
}

vehicles_clean["driver_sex_label"] = (
    vehicles_clean["sex_of_driver"].map(sex_map)
)

print("Driver sex labels created!")

print(
    vehicles_clean[
        ["sex_of_driver", "driver_sex_label"]
    ]
    .value_counts()
    .sort_index()
)

print(
    "\nUnmapped driver sex values:",
    vehicles_clean["driver_sex_label"].isna().sum()
)

Driver sex labels created!
sex_of_driver  driver_sex_label            
-1             Data missing or out of range      4307
 1             Male                            112413
 2             Female                           47045
 3             Not known                        20183
Name: count, dtype: int64

Unmapped driver sex values: 0


In [24]:
print("VEHICLE TYPE CODES")

print(
    vehicles_clean["vehicle_type"]
    .value_counts()
    .sort_index()
)

VEHICLE TYPE CODES
vehicle_type
-1          9
 1      17021
 2        813
 3       7876
 4       1792
 5       4976
 8       3614
 9     123212
 10       296
 11      3023
 16        78
 17       357
 18        28
 19     12555
 20       884
 21      2633
 22       346
 23       602
 33      1469
 90       770
 97       636
 98       673
 99       285
Name: count, dtype: int64


In [25]:
vehicle_related_columns = [
    col for col in vehicles_clean.columns
    if "vehicle" in col.lower()
    or "propulsion" in col.lower()
]

print("Vehicle-related columns:\n")

for col in vehicle_related_columns:
    print(col)

Vehicle-related columns:

vehicle_reference
vehicle_type
vehicle_manoeuvre_historic
vehicle_manoeuvre
vehicle_direction_from
vehicle_direction_to
vehicle_location_restricted_lane_historic
vehicle_location_restricted_lane
vehicle_leaving_carriageway
vehicle_left_hand_drive
propulsion_code
age_of_vehicle


In [26]:
print("PROPULSION CODE COUNTS")

print(
    vehicles_clean["propulsion_code"]
    .value_counts(dropna=False)
    .sort_index()
)

PROPULSION CODE COUNTS
propulsion_code
-1     38869
 1     74024
 2     50184
 3      6060
 5        64
 6         5
 7        74
 8     14193
 9         1
 10        1
 12      473
Name: count, dtype: int64


In [27]:
vehicle_propulsion_check = pd.crosstab(
    vehicles_clean["vehicle_type"],
    vehicles_clean["propulsion_code"]
)

display(vehicle_propulsion_check)

propulsion_code,-1,1,2,3,5,6,7,8,9,10,12
vehicle_type,,,,,,,,,,,
-1,9,0,0,0,0,0,0,0,0,0,0
1,17021,0,0,0,0,0,0,0,0,0,0
2,108,673,0,32,0,0,0,0,0,0,0
3,380,7436,1,59,0,0,0,0,0,0,0
4,177,1606,0,9,0,0,0,0,0,0,0
5,182,4787,4,1,0,1,0,1,0,0,0
8,366,146,1040,351,0,0,17,1679,0,0,15
9,13149,59191,33154,4711,4,4,56,12489,1,1,452
10,167,0,129,0,0,0,0,0,0,0,0


In [28]:
vehicle_type_map = {
    -1: "Data missing or out of range",
    1: "Pedal cycle",
    2: "Motorcycle 50cc and under",
    3: "Motorcycle 51cc-125cc",
    4: "Motorcycle 126cc-500cc",
    5: "Motorcycle over 500cc",
    8: "Taxi / Private hire car",
    9: "Car",
    10: "Minibus",
    11: "Bus or coach",
    16: "Ridden horse",
    17: "Agricultural vehicle",
    18: "Tram / Light rail",
    19: "Van / Goods vehicle 3.5t and under",
    20: "Goods vehicle over 3.5t to under 7.5t",
    21: "Goods vehicle 7.5t and over",
    22: "Mobility scooter",
    23: "Electric motorcycle",
    33: "Personal powered transporter",
    90: "Other vehicle",
    97: "Motorcycle - cc unknown",
    98: "Goods vehicle - unknown weight",
    99: "Unknown"
}

vehicles_clean["vehicle_type_label"] = (
    vehicles_clean["vehicle_type"].map(vehicle_type_map)
)

print("Vehicle type labels created!")

print(
    vehicles_clean["vehicle_type_label"]
    .value_counts(dropna=False)
)

print(
    "\nUnmapped vehicle type values:",
    vehicles_clean["vehicle_type_label"].isna().sum()
)

print(
    "Total vehicle records:",
    len(vehicles_clean)
)

Vehicle type labels created!
vehicle_type_label
Car                                      123212
Pedal cycle                               17021
Van / Goods vehicle 3.5t and under        12555
Motorcycle 51cc-125cc                      7876
Motorcycle over 500cc                      4976
Taxi / Private hire car                    3614
Bus or coach                               3023
Goods vehicle 7.5t and over                2633
Motorcycle 126cc-500cc                     1792
Personal powered transporter               1469
Goods vehicle over 3.5t to under 7.5t       884
Motorcycle 50cc and under                   813
Other vehicle                               770
Goods vehicle - unknown weight              673
Motorcycle - cc unknown                     636
Electric motorcycle                         602
Agricultural vehicle                        357
Mobility scooter                            346
Minibus                                     296
Unknown                                 

In [29]:
print("VEHICLE CLEANING VALIDATION")
print("-" * 40)

print("Rows:", len(vehicles_clean))
print("Columns:", vehicles_clean.shape[1])

print("\nMissing collision IDs:",
      vehicles_clean["collision_index"].isna().sum())

print("Unmatched collision IDs:",
      (~vehicles_clean["collision_index"]
       .isin(collisions_clean["collision_index"])).sum())

print("\nUnmapped vehicle types:",
      vehicles_clean["vehicle_type_label"].isna().sum())

print("Unmapped driver sex:",
      vehicles_clean["driver_sex_label"].isna().sum())

print("\nUnknown driver ages:",
      vehicles_clean["age_of_driver_clean"].isna().sum())

print("Original rows:",
      len(vehicles))

print("Rows after cleaning:",
      len(vehicles_clean))

VEHICLE CLEANING VALIDATION
----------------------------------------
Rows: 183948
Columns: 36

Missing collision IDs: 0
Unmatched collision IDs: 0

Unmapped vehicle types: 0
Unmapped driver sex: 0

Unknown driver ages: 28262
Original rows: 183948
Rows after cleaning: 183948


In [30]:
casualty_columns = [
    "casualty_class",
    "sex_of_casualty",
    "casualty_severity",
    "casualty_type",
    "pedestrian_location",
    "pedestrian_movement"
]

for column in casualty_columns:
    if column in casualties_clean.columns:
        print(f"\n{column}")
        print(
            casualties_clean[column]
            .value_counts(dropna=False)
            .sort_index()
        )
    else:
        print(f"\n{column}: COLUMN NOT FOUND")


casualty_class
casualty_class
1    83976
2    23929
3    19978
Name: count, dtype: int64

sex_of_casualty
sex_of_casualty
-1     1320
 1    77637
 2    48874
 9       52
Name: count, dtype: int64

casualty_severity
casualty_severity
1     1538
2    27758
3    98587
Name: count, dtype: int64

casualty_type
casualty_type
-1         2
 0     19978
 1     16042
 2       732
 3      7320
 4      1662
 5      4882
 8      1599
 9     65921
 10      233
 11     2036
 16       65
 17       78
 18       16
 19     3931
 20      202
 21      401
 22      271
 23      481
 33     1227
 90      305
 97      370
 98      114
 99       15
Name: count, dtype: int64

pedestrian_location
pedestrian_location
-1         43
 0     107900
 1       3379
 2         79
 3         28
 4       1078
 5       7213
 6       2628
 7        109
 8       1462
 9       2106
 10      1858
Name: count, dtype: int64

pedestrian_movement
pedestrian_movement
-1        41
 0    107899
 1      6322
 2       880
 3      4170

In [31]:
casualty_severity_map = {
    1: "Fatal",
    2: "Serious",
    3: "Slight"
}

casualties_clean["casualty_severity_label"] = (
    casualties_clean["casualty_severity"].map(casualty_severity_map)
)

print("Casualty severity labels created!")

print(
    casualties_clean[
        ["casualty_severity", "casualty_severity_label"]
    ]
    .value_counts()
    .sort_index()
)

print(
    "\nUnmapped casualty severity values:",
    casualties_clean["casualty_severity_label"].isna().sum()
)

print(
    "Total casualty records:",
    len(casualties_clean)
)

Casualty severity labels created!
casualty_severity  casualty_severity_label
1                  Fatal                       1538
2                  Serious                    27758
3                  Slight                     98587
Name: count, dtype: int64

Unmapped casualty severity values: 0
Total casualty records: 127883


In [32]:
casualty_class_map = {
    1: "Driver or rider",
    2: "Passenger",
    3: "Pedestrian"
}

casualties_clean["casualty_class_label"] = (
    casualties_clean["casualty_class"].map(casualty_class_map)
)

print("Casualty class labels created!")

print(
    casualties_clean[
        ["casualty_class", "casualty_class_label"]
    ]
    .value_counts()
    .sort_index()
)

print(
    "\nUnmapped casualty class values:",
    casualties_clean["casualty_class_label"].isna().sum()
)

print(
    "Total casualty records:",
    len(casualties_clean)
)

Casualty class labels created!
casualty_class  casualty_class_label
1               Driver or rider         83976
2               Passenger               23929
3               Pedestrian              19978
Name: count, dtype: int64

Unmapped casualty class values: 0
Total casualty records: 127883


In [33]:
casualty_sex_map = {
    -1: "Data missing or out of range",
    1: "Male",
    2: "Female",
    9: "Unknown"
}

casualties_clean["casualty_sex_label"] = (
    casualties_clean["sex_of_casualty"].map(casualty_sex_map)
)

print("Casualty sex labels created!")

print(
    casualties_clean[
        ["sex_of_casualty", "casualty_sex_label"]
    ]
    .value_counts()
    .sort_index()
)

print(
    "\nUnmapped casualty sex values:",
    casualties_clean["casualty_sex_label"].isna().sum()
)

print(
    "Total casualty records:",
    len(casualties_clean)
)

Casualty sex labels created!
sex_of_casualty  casualty_sex_label          
-1               Data missing or out of range     1320
 1               Male                            77637
 2               Female                          48874
 9               Unknown                            52
Name: count, dtype: int64

Unmapped casualty sex values: 0
Total casualty records: 127883


In [34]:
print("CASUALTY CLEANING VALIDATION")
print("-" * 40)

print("Rows:", len(casualties_clean))
print("Columns:", casualties_clean.shape[1])

print("\nMissing collision IDs:",
      casualties_clean["collision_index"].isna().sum())

print("Unmatched collision IDs:",
      (~casualties_clean["collision_index"]
       .isin(collisions_clean["collision_index"])).sum())

print("\nUnmapped casualty severity:",
      casualties_clean["casualty_severity_label"].isna().sum())

print("Unmapped casualty class:",
      casualties_clean["casualty_class_label"].isna().sum())

print("Unmapped casualty sex:",
      casualties_clean["casualty_sex_label"].isna().sum())

print("\nUnknown casualty ages:",
      casualties_clean["age_of_casualty_clean"].isna().sum())

print("\nOriginal rows:",
      len(casualties))

print("Rows after cleaning:",
      len(casualties_clean))

CASUALTY CLEANING VALIDATION
----------------------------------------
Rows: 127883
Columns: 29

Missing collision IDs: 0
Unmatched collision IDs: 0

Unmapped casualty severity: 0
Unmapped casualty class: 0
Unmapped casualty sex: 0

Unknown casualty ages: 2741

Original rows: 127883
Rows after cleaning: 127883


In [35]:
print("FINAL DATASET INTEGRITY CHECK")
print("=" * 45)

print("\nCOLLISIONS")
print("Rows:", len(collisions_clean))
print("Unique collision IDs:",
      collisions_clean["collision_index"].nunique())
print("Duplicate collision IDs:",
      collisions_clean["collision_index"].duplicated().sum())

print("\nVEHICLES")
print("Rows:", len(vehicles_clean))
print("Unique collision IDs:",
      vehicles_clean["collision_index"].nunique())
print("Unmatched collision IDs:",
      (~vehicles_clean["collision_index"]
       .isin(collisions_clean["collision_index"])).sum())

print("\nCASUALTIES")
print("Rows:", len(casualties_clean))
print("Unique collision IDs:",
      casualties_clean["collision_index"].nunique())
print("Unmatched collision IDs:",
      (~casualties_clean["collision_index"]
       .isin(collisions_clean["collision_index"])).sum())

print("\nROW PRESERVATION")
print("Collisions:",
      len(collisions), "→", len(collisions_clean))
print("Vehicles:",
      len(vehicles), "→", len(vehicles_clean))
print("Casualties:",
      len(casualties), "→", len(casualties_clean))

FINAL DATASET INTEGRITY CHECK

COLLISIONS
Rows: 101525
Unique collision IDs: 101525
Duplicate collision IDs: 0

VEHICLES
Rows: 183948
Unique collision IDs: 101525
Unmatched collision IDs: 0

CASUALTIES
Rows: 127883
Unique collision IDs: 101525
Unmatched collision IDs: 0

ROW PRESERVATION
Collisions: 101525 → 101525
Vehicles: 183948 → 183948
Casualties: 127883 → 127883


In [36]:
# Remove temporary helper column before export
if "time_clean" in collisions_clean.columns:
    collisions_clean = collisions_clean.drop(columns=["time_clean"])

print("Collision dataset prepared for export!")

print("Final rows:", collisions_clean.shape[0])
print("Final columns:", collisions_clean.shape[1])

print(
    "time_clean still present:",
    "time_clean" in collisions_clean.columns
)

print(
    "\nUseful time columns:",
    [
        col for col in
        ["date", "time", "hour", "time_of_day", "peak_period"]
        if col in collisions_clean.columns
    ]
)

Collision dataset prepared for export!
Final rows: 101525
Final columns: 61
time_clean still present: False

Useful time columns: ['date', 'time', 'hour', 'time_of_day', 'peak_period']


In [37]:
PROCESSED_PATH = Path("../data/processed")

# Create the folder if it does not already exist
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

# Export cleaned datasets
collisions_clean.to_csv(
    PROCESSED_PATH / "collisions_clean.csv",
    index=False
)

vehicles_clean.to_csv(
    PROCESSED_PATH / "vehicles_clean.csv",
    index=False
)

casualties_clean.to_csv(
    PROCESSED_PATH / "casualties_clean.csv",
    index=False
)

print("Cleaned datasets exported successfully!")

print("\nExported files:")
print(PROCESSED_PATH / "collisions_clean.csv")
print(PROCESSED_PATH / "vehicles_clean.csv")
print(PROCESSED_PATH / "casualties_clean.csv")

Cleaned datasets exported successfully!

Exported files:
..\data\processed\collisions_clean.csv
..\data\processed\vehicles_clean.csv
..\data\processed\casualties_clean.csv


In [38]:
# Read the exported files back from disk
collisions_exported = pd.read_csv(
    PROCESSED_PATH / "collisions_clean.csv"
)

vehicles_exported = pd.read_csv(
    PROCESSED_PATH / "vehicles_clean.csv"
)

casualties_exported = pd.read_csv(
    PROCESSED_PATH / "casualties_clean.csv"
)

print("EXPORTED FILE VERIFICATION")
print("=" * 40)

print(
    "Collisions:",
    collisions_exported.shape
)

print(
    "Vehicles:",
    vehicles_exported.shape
)

print(
    "Casualties:",
    casualties_exported.shape
)

print("\nExpected shapes:")
print("Collisions:", collisions_clean.shape)
print("Vehicles:", vehicles_clean.shape)
print("Casualties:", casualties_clean.shape)

print("\nVerification:")
print(
    "Collisions match:",
    collisions_exported.shape == collisions_clean.shape
)

print(
    "Vehicles match:",
    vehicles_exported.shape == vehicles_clean.shape
)

print(
    "Casualties match:",
    casualties_exported.shape == casualties_clean.shape
)

C:\Users\shiva\AppData\Local\Temp\ipykernel_16136\3449434334.py:6: DtypeWarning: Columns (27: generic_make_model) have mixed types. Specify dtype option on import or set low_memory=False.
  vehicles_exported = pd.read_csv(


EXPORTED FILE VERIFICATION
Collisions: (101525, 61)
Vehicles: (183948, 36)
Casualties: (127883, 29)

Expected shapes:
Collisions: (101525, 61)
Vehicles: (183948, 36)
Casualties: (127883, 29)

Verification:
Collisions match: True
Vehicles match: True
Casualties match: True
